In [44]:
from dataclasses import dataclass, asdict
import json

@dataclass
class Ride:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    pulocationid: int
    dolocationid: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float

def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=str(row['lpep_pickup_datetime']),
        lpep_dropoff_datetime=str(row['lpep_dropoff_datetime']),
        pulocationid=int(row['PULocationID']),
        dolocationid=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [31]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"


columns = ['lpep_pickup_datetime','lpep_dropoff_datetime','PULocationID','DOLocationID','passenger_count','trip_distance','tip_amount','total_amount']
df = pd.read_parquet(url, columns=columns) #columns=columns
df.head()
len(df)

49416

In [32]:
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'
topic_name = 'green-trips'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [47]:
# Q2
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=asdict(ride))
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

ValueError: cannot convert float NaN to integer

In [35]:
# Q3

from kafka import KafkaConsumer

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    value_deserializer=ride_deserializer
)

data = [] 
count = 0

for message in consumer:
    ride = message.value
    data.append(ride)
    count += 1
    if count >= 49416:
        break
    
consumer.close()

df_consume = pd.DataFrame(data)
len(df_consume.query('trip_distance > 5'))

8506

In [ ]:
# Q4
import random

try:
    while True:
        # ~20% chance of a late event (3-10 seconds old)
        if random.random() < 0.2:
            delay = random.randint(3, 10)
            ride = make_ride(delay_seconds=delay)
            ts = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000, tz=timezone.utc)
            print(f"  LATE ({delay}s) -> PU={ride.PULocationID} ts={ts:%H:%M:%S}")
        else:
            ride = make_ride()
            ts = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000, tz=timezone.utc)
            print(f"  on time   -> PU={ride.PULocationID} ts={ts:%H:%M:%S}")

        producer.send(topic_name, value=ride)
        count += 1
        time.sleep(0.5)

except KeyboardInterrupt:
    producer.flush()
    print(f"\nSent {count} events")

1000

In [ ]:
CREATE TABLE IF NOT EXISTS processed_events_aggregated (
    window_start TIMESTAMP(3) NOT NULL,
    pulocationid INTEGER NOT NULL,  -- Обратите внимание на регистр!
    num_trips BIGINT,
    PRIMARY KEY (window_start, pulocationid)
);

CREATE TABLE IF NOT EXISTS processed_events_aggregated_2 (
    window_start TIMESTAMP(3) NOT NULL,
    pulocationid INTEGER NOT NULL,  -- Обратите внимание на регистр!
    num_trips BIGINT,
    PRIMARY KEY (window_start, pulocationid)
);

CREATE TABLE IF NOT EXISTS processed_events_aggregated_3 (
    window_start TIMESTAMP(3) NOT NULL,
    total_tip_amount FLOAT NOT NULL,  -- Обратите внимание на регистр!
    PRIMARY KEY (window_start)
);


SELECT pulocationid, num_trips
FROM processed_events_aggregated
ORDER BY num_trips DESC
LIMIT 3;

select max(t.num_trips) from(
select pulocationid, num_trips, sum(num_trips)
from processed_events_aggregated
group by
	pulocationid, window_start)t
	
SELECT top 1 window_start, total_tip_amount
FROM public.processed_events_aggregated_3 order by total_tip_amount desc
	

1000